# Pre Trained Models

### Cell 1: Import Libraries and Model Definitions

In [1]:
# Cell 1: Import Libraries and Model Definitions
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import os
import glob
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Model Definitions
class OCTResNet18(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(OCTResNet18, self).__init__()
        self.backbone = models.resnet18(pretrained=pretrained)
        
        # Modify first conv layer for single channel input
        original_conv1 = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Initialize with pretrained weights (average across RGB channels)
        if pretrained:
            with torch.no_grad():
                self.backbone.conv1.weight = nn.Parameter(
                    original_conv1.weight.mean(dim=1, keepdim=True)
                )
        
        # Replace final layer
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.backbone(x)

class OCTDenseNet121(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(OCTDenseNet121, self).__init__()
        self.backbone = models.densenet121(pretrained=pretrained)
        
        # Modify first conv layer for single channel input
        original_conv0 = self.backbone.features.conv0
        self.backbone.features.conv0 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        
        # Initialize with pretrained weights
        if pretrained:
            with torch.no_grad():
                self.backbone.features.conv0.weight = nn.Parameter(
                    original_conv0.weight.mean(dim=1, keepdim=True)
                )
        
        # Replace classifier
        in_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.backbone(x)

class OCTVGG16(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(OCTVGG16, self).__init__()
        self.backbone = models.vgg16(pretrained=pretrained)
        
        # Modify first conv layer for single channel input
        original_features = self.backbone.features
        new_features = nn.Sequential()
        
        # Replace first conv layer
        new_features.add_module('conv0', nn.Conv2d(1, 64, kernel_size=3, padding=1))
        if pretrained:
            with torch.no_grad():
                new_features.conv0.weight = nn.Parameter(
                    original_features[0].weight.mean(dim=1, keepdim=True)
                )
                new_features.conv0.bias = original_features[0].bias
        
        # Add remaining layers
        for i, layer in enumerate(original_features[1:], 1):
            new_features.add_module(f'layer{i}', layer)
        
        self.backbone.features = new_features
        
        # Replace classifier
        in_features = self.backbone.classifier[6].in_features
        self.backbone.classifier[6] = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        return self.backbone(x)

class OCTMobileNet(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super(OCTMobileNet, self).__init__()
        self.backbone = models.mobilenet_v2(pretrained=pretrained)
        
        # Modify first conv layer for single channel input
        original_conv = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1, bias=False)
        
        # Initialize with pretrained weights
        if pretrained:
            with torch.no_grad():
                self.backbone.features[0][0].weight = nn.Parameter(
                    original_conv.weight.mean(dim=1, keepdim=True)
                )
        
        # Replace classifier
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

Using device: cpu


### Cell 2: Dataset and Data Loading Functions

In [2]:
# Cell 2: Dataset and Data Loading Functions
class FrameLevelDataset(Dataset):
    """Dataset for frame-level training (each frame is a separate sample)"""
    def __init__(self, images, labels, patient_ids=None):
        self.images = images
        self.labels = labels
        self.patient_ids = patient_ids
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        # Ensure image has proper shape (1, 64, 64)
        if len(image.shape) == 2:
            image = np.expand_dims(image, axis=0)
        
        image = torch.FloatTensor(image)
        return image, label

def load_and_preprocess_image(img_path, img_size=64):
    """Load and preprocess a single image"""
    img = Image.open(img_path)
    if img.mode != 'L':
        img = img.convert('L')
    img = img.resize((img_size, img_size))
    img_array = np.array(img, dtype=np.float32) / 255.0
    return img_array

def load_frame_data_with_patient_info(data_dir, img_size=64):
    """
    Load frame-level data with patient information to prevent data leakage
    """
    images = []
    labels = []
    patient_ids = []
    image_paths = []
    
    class_mapping = {'MS': 1, 'Control': 0}
    
    for class_name, class_label in class_mapping.items():
        class_dir = os.path.join(data_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"Warning: {class_dir} does not exist")
            continue
            
        image_files = glob.glob(os.path.join(class_dir, "*.png")) + \
                     glob.glob(os.path.join(class_dir, "*.jpg")) + \
                     glob.glob(os.path.join(class_dir, "*.jpeg"))
        image_files.sort()
        
        print(f"Loading {class_name} data: {len(image_files)} frames")
        
        for img_path in image_files:
            try:
                img = load_and_preprocess_image(img_path, img_size)
                images.append(img)
                labels.append(class_label)
                image_paths.append(img_path)
                
                # Extract patient ID from filename
                filename = os.path.basename(img_path)
                patient_id = filename.split('_')[0]
                patient_ids.append(patient_id)
                
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                continue
    
    return np.array(images), np.array(labels), patient_ids, image_paths

### Cell 3: Training and Evaluation Classes

In [3]:
# Cell 3: Training and Evaluation Classes
class OCTTrainer:
    def __init__(self, model, device, criterion, optimizer):
        self.model = model
        self.device = device
        self.criterion = criterion
        self.optimizer = optimizer
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        
    def train_epoch(self, dataloader):
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for data, target in dataloader:
            data, target = data.to(self.device), target.to(self.device)
            
            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
        
        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100. * correct / total
        return epoch_loss, epoch_acc
    
    def validate_epoch(self, dataloader):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(self.device), target.to(self.device)
                output = self.model(data)
                loss = self.criterion(output, target)
                
                running_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()
        
        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100. * correct / total
        return epoch_loss, epoch_acc
    
    def train(self, train_loader, val_loader, epochs, verbose=True):
        for epoch in range(epochs):
            train_loss, train_acc = self.train_epoch(train_loader)
            val_loss, val_acc = self.validate_epoch(val_loader)
            
            self.train_losses.append(train_loss)
            self.val_losses.append(val_loss)
            self.train_accuracies.append(train_acc)
            self.val_accuracies.append(val_acc)
            
            if verbose:
                print(f'Epoch {epoch+1}/{epochs}:')
                print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
                print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
                print('-' * 50)

class ModelEvaluator:
    @staticmethod
    def evaluate_model(model, dataloader, device):
        model.eval()
        all_predictions = []
        all_targets = []
        all_probabilities = []
        
        with torch.no_grad():
            for data, target in dataloader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                probabilities = torch.softmax(output, dim=1)
                _, predicted = torch.max(output, 1)
                
                all_predictions.extend(predicted.cpu().numpy())
                all_targets.extend(target.cpu().numpy())
                all_probabilities.extend(probabilities.cpu().numpy())
        
        return np.array(all_predictions), np.array(all_targets), np.array(all_probabilities)
    
    @staticmethod
    def calculate_metrics(predictions, targets, probabilities):
        accuracy = accuracy_score(targets, predictions)
        precision = precision_score(targets, predictions, average='binary', zero_division=0)
        recall = recall_score(targets, predictions, average='binary', zero_division=0)
        f1 = f1_score(targets, predictions, average='binary', zero_division=0)
        
        # Calculate Sensitivity and Specificity
        cm = confusion_matrix(targets, predictions)
        
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        else:
            sensitivity = recall
            specificity = 0
            if len(np.unique(targets)) == 2:
                tn = np.sum((predictions == 0) & (targets == 0))
                fp = np.sum((predictions == 1) & (targets == 0))
                fn = np.sum((predictions == 0) & (targets == 1))
                tp = np.sum((predictions == 1) & (targets == 1))
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        auc = roc_auc_score(targets, probabilities[:, 1]) if len(np.unique(targets)) > 1 else 0.5
        
        metrics = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'auc': auc,
            'confusion_matrix': cm
        }
        
        return metrics
    
    @staticmethod
    def print_metrics(metrics, title="Model Evaluation Metrics"):
        print(f"\n{title}")
        print("=" * 70)
        print(f"Accuracy:     {metrics['accuracy']:.4f}")
        print(f"Precision:    {metrics['precision']:.4f}")
        print(f"Recall:       {metrics['recall']:.4f}")
        print(f"F1-Score:     {metrics['f1_score']:.4f}")
        print(f"Sensitivity:  {metrics['sensitivity']:.4f}")
        print(f"Specificity:  {metrics['specificity']:.4f}")
        print(f"AUC:          {metrics['auc']:.4f}")
        print("\nConfusion Matrix:")
        print(metrics['confusion_matrix'])

### Cell 4: Patient-Level LOOCV for All Models

In [4]:
# Cell 4: Patient-Level LOOCV for All Models
def run_patient_level_loocv_all_models(data_dir, model_name='resnet18', img_size=64, epochs=10, batch_size=32):
    """
    Run Leave-One-Patient-Out Cross Validation for all pre-trained models
    """
    print(f"=== PATIENT-LEVEL LOOCV FOR {model_name.upper()} ===")
    
    # Initialize model
    if model_name == 'resnet18':
        model = OCTResNet18(num_classes=2, pretrained=True)
    elif model_name == 'densenet121':
        model = OCTDenseNet121(num_classes=2, pretrained=True)
    elif model_name == 'vgg16':
        model = OCTVGG16(num_classes=2, pretrained=True)
    elif model_name == 'mobilenet':
        model = OCTMobileNet(num_classes=2, pretrained=True)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    model.to(device)
    print(f"Model: {model_name}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Load data with patient information
    images, labels, patient_ids, image_paths = load_frame_data_with_patient_info(data_dir, img_size)
    
    if len(images) == 0:
        print("No data loaded!")
        return None, None
    
    # Get unique patients
    unique_patients = list(set(patient_ids))
    print(f"Found {len(unique_patients)} unique patients")
    print(f"Total frames: {len(images)}")
    
    # Store results
    all_predictions = []
    all_targets = []
    all_probabilities = []
    all_test_patients = []
    patient_level_results = {}
    
    print(f"\nRunning Leave-One-Patient-Out Cross Validation...")
    
    for fold, test_patient in enumerate(unique_patients):
        print(f"\nFold {fold + 1}/{len(unique_patients)} - Testing Patient: {test_patient}")
        
        # Split data by patient
        train_indices = [i for i, pid in enumerate(patient_ids) if pid != test_patient]
        test_indices = [i for i, pid in enumerate(patient_ids) if pid == test_patient]
        
        if len(train_indices) == 0 or len(test_indices) == 0:
            print(f"  Skipping - no training or test data")
            continue
        
        # Get train and test data
        X_train = images[train_indices]
        y_train = labels[train_indices]
        X_test = images[test_indices]
        y_test = labels[test_indices]
        
        # Further split training data for validation
        train_patients = list(set([patient_ids[i] for i in train_indices]))
        if len(train_patients) > 1:
            val_patient = train_patients[0]
            val_indices = [i for i, pid in enumerate(patient_ids) if pid == val_patient and i in train_indices]
            train_indices_final = [i for i in train_indices if patient_ids[i] != val_patient]
            
            X_train_final = images[train_indices_final]
            y_train_final = labels[train_indices_final]
            X_val = images[val_indices]
            y_val = labels[val_indices]
        else:
            X_train_final, X_val, y_train_final, y_val = X_train, X_train, y_train, y_train
        
        print(f"  Training frames: {len(X_train_final)}")
        print(f"  Validation frames: {len(X_val)}")
        print(f"  Test frames: {len(X_test)}")
        
        # Create datasets and data loaders
        train_dataset = FrameLevelDataset(X_train_final, y_train_final)
        val_dataset = FrameLevelDataset(X_val, y_val)
        test_dataset = FrameLevelDataset(X_test, y_test)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        
        # Reinitialize model for each fold
        if model_name == 'resnet18':
            model = OCTResNet18(num_classes=2, pretrained=True)
        elif model_name == 'densenet121':
            model = OCTDenseNet121(num_classes=2, pretrained=True)
        elif model_name == 'vgg16':
            model = OCTVGG16(num_classes=2, pretrained=True)
        elif model_name == 'mobilenet':
            model = OCTMobileNet(num_classes=2, pretrained=True)
        
        model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
        
        # Train model
        trainer = OCTTrainer(model, device, criterion, optimizer)
        trainer.train(train_loader, val_loader, epochs, verbose=False)
        
        # Evaluate on test set
        predictions, targets, probabilities = ModelEvaluator.evaluate_model(model, test_loader, device)
        
        if len(targets) > 0:
            # Calculate metrics for this patient
            patient_metrics = ModelEvaluator.calculate_metrics(predictions, targets, probabilities)
            
            # Store results
            all_predictions.extend(predictions)
            all_targets.extend(targets)
            all_probabilities.extend(probabilities)
            all_test_patients.extend([test_patient] * len(targets))
            
            # Store patient-level results
            patient_level_results[test_patient] = {
                'metrics': patient_metrics,
                'predictions': predictions,
                'targets': targets,
                'probabilities': probabilities,
                'num_frames': len(targets)
            }
            
            print(f"  Patient Results:")
            print(f"    Accuracy:    {patient_metrics['accuracy']:.4f}")
            print(f"    Precision:   {patient_metrics['precision']:.4f}")
            print(f"    F1-Score:    {patient_metrics['f1_score']:.4f}")
            print(f"    Sensitivity: {patient_metrics['sensitivity']:.4f}")
            print(f"    Specificity: {patient_metrics['specificity']:.4f}")
            print(f"    AUC:         {patient_metrics['auc']:.4f}")
    
    # Calculate overall metrics
    if len(all_targets) > 0:
        overall_metrics = ModelEvaluator.calculate_metrics(
            np.array(all_predictions), 
            np.array(all_targets), 
            np.array(all_probabilities)
        )
        
        print(f"\n{'='*70}")
        print(f"OVERALL LOOCV RESULTS - {model_name.upper()}")
        print(f"{'='*70}")
        ModelEvaluator.print_metrics(overall_metrics, f"Overall {model_name.upper()} LOOCV Metrics")
        
        # Print patient-level summary
        print(f"\n{'='*70}")
        print("PATIENT-LEVEL SUMMARY")
        print(f"{'='*70}")
        print(f"{'Patient':<15} {'Frames':<8} {'Accuracy':<10} {'Precision':<10} {'F1-Score':<10} {'Sensitivity':<12} {'Specificity':<12} {'AUC':<8}")
        print("-" * 95)
        
        for patient, results in patient_level_results.items():
            metrics = results['metrics']
            print(f"{patient:<15} {results['num_frames']:<8} {metrics['accuracy']:.4f}     "
                  f"{metrics['precision']:.4f}     {metrics['f1_score']:.4f}     "
                  f"{metrics['sensitivity']:.4f}       {metrics['specificity']:.4f}       "
                  f"{metrics['auc']:.4f}")
        
        return overall_metrics, patient_level_results
    
    print("No valid results collected from LOOCV!")
    return None, None

### Cell 5: Main Function to Compare All Models

In [8]:
# Cell 5: Main Function to Compare All Models
def main():
    """Main function to run all pre-trained models with Patient-Level LOOCV"""
    print("OCT Classification - Pre-trained Models Comparison")
    print("=" * 70)
    
    # UPDATE THIS PATH to your frame-level data directory
    FRAME_LEVEL_DATA_PATH = "/home/tanvirdell3/Downloads/caserel-master/Publically available Dataset/Segmentation Sharpen filter"  # ⬅️ UPDATE THIS PATH
    
    if not os.path.exists(FRAME_LEVEL_DATA_PATH):
        print(f"Data path {FRAME_LEVEL_DATA_PATH} does not exist!")
        return
    #'resnet18', 'densenet121', 'vgg16', 'mobilenet'
    models_to_test = ['vgg16'] 
    all_results = {}
    
    for model_name in models_to_test:
        print(f"\n{'='*70}")
        print(f"TESTING {model_name.upper()}")
        print(f"{'='*70}")
        
        overall_metrics, patient_results = run_patient_level_loocv_all_models(
            FRAME_LEVEL_DATA_PATH, 
            model_name=model_name, 
            epochs=20,  # Reduced for pre-trained models
            batch_size=16
        )
        
        if overall_metrics:
            all_results[model_name] = overall_metrics
    
    # Print final comparison
    print(f"\n{'='*80}")
    print("FINAL COMPARISON - ALL MODELS")
    print(f"{'='*80}")
    
    print(f"\n{'Model':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Sensitivity':<12} {'Specificity':<12} {'AUC':<10}")
    print("-" * 100)
    
    for model_name, metrics in all_results.items():
        print(f"{model_name:<15} {metrics['accuracy']:.4f}     "
              f"{metrics['precision']:.4f}     {metrics['recall']:.4f}     "
              f"{metrics['f1_score']:.4f}     {metrics['sensitivity']:.4f}        "
              f"{metrics['specificity']:.4f}        {metrics['auc']:.4f}")

# Run the main function
if __name__ == "__main__":
    main()

OCT Classification - Pre-trained Models Comparison

TESTING VGG16
=== PATIENT-LEVEL LOOCV FOR VGG16 ===
Model: vgg16
Total parameters: 134,267,586
Loading MS data: 147 frames
Loading Control data: 98 frames
Found 35 unique patients
Total frames: 245

Running Leave-One-Patient-Out Cross Validation...

Fold 1/35 - Testing Patient: hc11
  Training frames: 231
  Validation frames: 7
  Test frames: 7
  Patient Results:
    Accuracy:    0.0000
    Precision:   0.0000
    F1-Score:    0.0000
    Sensitivity: 0.0000
    Specificity: 0.0000
    AUC:         0.5000

Fold 2/35 - Testing Patient: hc14
  Training frames: 231
  Validation frames: 7
  Test frames: 7
  Patient Results:
    Accuracy:    0.0000
    Precision:   0.0000
    F1-Score:    0.0000
    Sensitivity: 0.0000
    Specificity: 0.0000
    AUC:         0.5000

Fold 3/35 - Testing Patient: ms09
  Training frames: 231
  Validation frames: 7
  Test frames: 7
  Patient Results:
    Accuracy:    1.0000
    Precision:   1.0000
    F1-Score

### Done                                                                                                                                                         